In [1]:
%pip install numpy pandas matplotlib seaborn plotly nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd
from jedi.inference import lazy_value
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
items = pd.read_csv('olist_order_items_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
translations = pd.read_csv('product_category_name_translation.csv')
df_raw = (items.merge(orders, on='order_id', how='left')
           .merge(customers, on='customer_id', how='left')
           .merge(products, on='product_id', how='left')
           .merge(translations, on='product_category_name', how='left'))

df = df_raw.sample(n=10000, random_state=24).copy()
display(df.head(10))

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
83715,be1aff1e5ed8adabab251a979e96bdb9,3,4052517cac9e78357d895976124f6972,ce27a3cc3c8cc1ea79d11e561e9bebb6,2017-07-21 14:35:13,150.00,27.59,826c677b00d964c53329e5ea1b5bddab,delivered,2017-07-14 14:20:03,2017-07-14 14:35:13,2017-07-17 21:34:59,2017-07-27 16:44:30,2017-08-18 00:00:00,76d5f0e306da4dcc92bb8c4a00b4f891,63260,brejo santo,CE,eletronicos,31.0,736.0,3.0,800.0,35.0,30.0,40.0,electronics
70483,a0e058d95d9625fbb9a4bd69fd6d0996,1,119cfca51257edda1acecb4b8576ec48,cd68562d3f44870c08922d380acae552,2017-08-09 03:10:54,169.00,12.68,1d46f6ccc4985ed63a309848cad48bd5,delivered,2017-08-01 17:18:33,2017-08-03 03:10:54,2017-08-03 18:51:42,2017-08-09 23:07:40,2017-08-21 00:00:00,2bc00078081fa117b125f5bd5e145764,12606,lorena,SP,cool_stuff,59.0,1618.0,3.0,600.0,16.0,14.0,15.0,cool_stuff
28591,411863c544c533d3918d4e274ce334c8,1,e78506414d2563b6414cfd84b7fa0802,fe2032dab1a61af8794248c8196565c9,2017-12-06 12:30:29,152.00,18.34,a8fbdfdfcd4b5f2f67dcd87ca80add6c,delivered,2017-11-29 09:54:11,2017-11-30 13:31:21,2017-12-05 18:13:15,2018-01-03 21:21:59,2017-12-27 00:00:00,16b4e6a5fcbd35d8b50d07c047f30d17,60541,fortaleza,CE,perfumaria,58.0,827.0,1.0,460.0,20.0,14.0,20.0,perfumery
19801,2d736835b63820cfdc9c9e7eef22daf2,1,9bb8ca338e5588c361e34eae02e8fad6,620c87c171fb2a6dd6e8bb4dec959fc6,2017-11-21 02:55:51,59.90,14.01,c06dfa7240c4304b19b4246f3b107d38,delivered,2017-11-14 10:40:21,2017-11-15 02:56:21,2017-11-16 15:34:49,2017-11-17 20:11:36,2017-12-04 00:00:00,4249a1724d282a6356179a862fe02722,27253,volta redonda,RJ,beleza_saude,37.0,314.0,1.0,431.0,19.0,17.0,15.0,health_beauty
71453,a2f4de10fceff6c008efd19d2b99725f,1,cd7f7f00061ea28ae06afc6dc5786cf0,5a8e7d5003a1f221f9e1d6e411de7c23,2018-08-14 12:04:28,79.90,8.72,b66217448555073a1804c56d701fa4e1,delivered,2018-08-11 11:48:41,2018-08-11 12:04:28,2018-08-13 15:09:00,2018-08-15 17:48:37,2018-08-20 00:00:00,74635e4c481883885fbc9429b880df02,13015,campinas,SP,moveis_decoracao,33.0,1183.0,5.0,350.0,68.0,8.0,13.0,furniture_decor
45577,677c865b89aa97e7dbeb4bfbb50bbcfd,1,e2bab04d6a471ed7a3629fbe2a64b55a,beadbee30901a7f61d031b6b686095ad,2018-08-30 23:24:13,85.00,7.86,529d3f46584a8fa9027777a9c7d5a1fc,delivered,2018-08-14 23:08:33,2018-08-14 23:24:13,2018-08-23 12:51:00,2018-08-24 23:42:42,2018-08-31 00:00:00,e1881c7d99c20ca24a49cf18f115db40,2435,sao paulo,SP,perfumaria,51.0,950.0,1.0,300.0,16.0,19.0,14.0,perfumery
64106,92719473371fb9439b386d53d12470a8,1,dc52f0f5d3ec37a93eaf956cde4e5d2c,6560211a19b47992c3666cc44a7e94c0,2017-10-19 20:46:38,49.00,21.15,5944fcd0423637dc7497ef10187e073b,delivered,2017-10-13 19:25:59,2017-10-13 19:46:38,2017-10-16 20:12:41,2017-11-23 21:13:07,2017-11-06 00:00:00,c601f5196438b2836f6de978734b2829,49027,aracaju,SE,relogios_presentes,60.0,539.0,7.0,400.0,16.0,2.0,20.0,watches_gifts
14021,1fedda1b294263f50d8f0f78bf5c3e6a,1,7c2d0df8e5fbfa71d54a400b33a8502e,7e1fb0a3ebfb01ffb3a7dae98bf3238d,2017-05-10 21:25:17,116.00,16.51,08f68c9d6b891c565a87205b547d21a8,delivered,2017-05-04 21:15:50,2017-05-04 21:25:17,2017-05-05 16:26:50,2017-05-16 10:12:49,2017-05-31 00:00:00,fcbbe2856825553e6ed2bfbb252587fc,29730,baixo guandu,ES,beleza_saude,59.0,1515.0,1.0,138.0,12.0,9.0,18.0,health_beauty
44758,65c28faddc1bdb2d3d5de4f713de3f9a,1,a56fbff66c12c1ef311cac69e7f2d5fb,1025f0e2d44d7041d6cf58b6550e0bfa,2018-04-16 04:10:47,45.00,17.06,e8334d453a2049e7a529d772be66ad61,delivered,2018-04-06 12:54:10,2018-04-10 04:10:47,2018-04-11 20:22:01,2018-04-24 18:29:49,2018-05-04 00:00:00,c68a95b096e94c5c6e1f47c06d847171,41770,salv

In [4]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df = df.sort_values('order_purchase_timestamp').reset_index(drop=True)
df_initial = df.iloc[:9000].copy()
df_delta = df.iloc[9000:].copy()

targets = df_initial.drop_duplicates(subset=['customer_unique_id']).head(30).copy()

scd1_case = targets.iloc[0:10].copy()
scd1_case['order_status'] = 'delivered'

scd2_case = targets.iloc[10:20].copy()
scd2_case['customer_state'] = 'RJ'

duplicates = targets.iloc[20:30].copy()

df_delta_final = pd.concat([df_delta,scd1_case, scd2_case,duplicates], ignore_index=True)

df_initial.to_csv('initial_load.csv', index=False)
df_delta_final.to_csv('delta_load.csv', index=False)



In [5]:
display(df.columns)

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'customer_id',
       'order_status', 'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english'],
      dtype='str')

 1) Explain the purpose of the three-layer DWH architecture. Why shouldn't the BI team just connect Power BI directly to the Stage layer or the raw transactional source system?

Connecting Power BI directly to the raw or stage layer is a bad practice for three main reasons: it results in slow performance, exposes "dirty" data, and loses historical context because raw systems typically overwrite old records.

The three-layer architecture solves these issues:
- Stage Layer: A temporary landing zone used to quickly extract raw data without impacting the performance of the source systems.
- Core Layer: Cleanses the data, filters out duplicates, and manages historical changes in a normalized structure to ensure data integrity.
- Mart Layer: Denormalizes the core layer into Fact and Dimension tables. This structure is perfectly optimized for reporting tools, ensuring lightning-fast query performance and easy analysis.

2. Technical Reflection: For your Mart layer, you were asked to build a denormalized structure. Contrast this with the Core layer (3NF). Why do reporting tools like Power BI perform significantly better with a Star Schema rather than a highly normalized relational model?

The core layer uses a highly normalized 3NF model to eliminate data redundancy, ensure strict data integrity, and efficiently process updates. However, this structure fragments the data across many tables, requiring complex and slow JOIN operations to piece the information back together. The mart layer uses a denormalized Star Schema, centralizing transactions into a Fact table surrounded by flat, wide Dimension tables.

Reporting tools perform significantly better with a Star Schema for two main reasons:
- Fewer JOINs: Analytical queries only need a single join from the Fact table to the relevant Dimensions. This drastically reduces query execution time compared to navigating the complex web of a 3NF model.
- VertiPaq Engine Optimization: Power BI operates on a columnar in-memory database engine. This engine is specifically designed to compress and instantly filter the repeating text values found in wide, denormalized dimension tables, resulting in lightning-fast dashboard performance.

3. Business Reflection: Give a real-world business example from your specific domain (Fintech, HR, Agro, etc.) where overwriting an old attribute (using SCD Type 1 instead of SCD Type 2) would result in incorrect historical reporting.


Example from the E-commerce

Company analyzing revenue by geographic region. In 2023, a highly active customer lives in New York and generates $10,000 in sales. In 2024, this customer relocates to California. If the data warehouse uses SCD Type 1, the customer's location in the database is simply updated from New York to California.

The consequence: When a regional manager generates a historical revenue report for 2023, that $10,000 is retroactively shifted to California. This corrupts the historical data—making New York's past financial performance look worse than it actually was, and artificially inflating California's numbers.

Using SCD Type 2 prevents this by closing the New York record and opening a new California record. This ensures that the 2023 transactions remain permanently tied to New York, while only new purchases are attributed to California, preserving accurate historical reporting.

4. Why is it impossible to implement SCD Type 2 correctly using only the original Natural Key (e.g., User ID, Client ID) from the CSV file? Explain the role of a Surrogate Key in this process.

It is impossible to implement SCD Type 2 using only a Natural Key because a Primary Key in a relational database must be strictly unique.

SCD Type 2 works by keeping the old record and inserting a brand-new row to capture the updated information. If the Natural Key like Client ID is used as the Primary Key, the database will block the new row from being inserted, throwing a duplicate key constraint error.

A Surrogate Key is an artificial, system-generated identifier that acts as the actual Primary Key for the Dimension table. It solves this problem by allowing multiple rows with the same Natural Key to exist in the table. Each row represents a specific version of that customer in time, and each gets its own unique Surrogate Key. The Fact table then connects to this Surrogate Key, ensuring every transaction is tied to the exact historical profile of the customer at the moment the purchase was made.